# **Extractive Summarisation using TextRank**

This notebook implements an extractive summarisation baseline for policy documents.

The goal is to automatically generate concise summaries by selecting the most important sentences from the original text.

We use the TextRank algorithm, which ranks sentences based on their similarity and importance within the document.

The performance of the model is evaluated using ROUGE metrics, and additional experiments are conducted to analyse the effect of the sentence selection parameter (top_k).

---

In [21]:
# =========================
# 1. Install required packages
# =========================
# Install necessary libraries for data processing, NLP, and evaluation

!pip install -q pandas numpy scikit-learn nltk networkx rouge-score

In [22]:
# =========================
# 2. Import libraries
# =========================
# Import required libraries for data handling, NLP processing, and evaluation

import os
import json
import re
import pandas as pd
import numpy as np
import nltk
import networkx as nx

from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [23]:
# =========================
# 3. Load dataset
# =========================
# Load the ToS;DR dataset containing policy documents and reference summaries

from google.colab import drive
drive.mount('/content/drive')
DATA_PATH = "/content/drive/MyDrive/tosdr_annotated_v1.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

# Convert JSON to DataFrame
df = pd.DataFrame(data).T.reset_index()

# Remove duplicated uid column if exists
if "uid" in df.columns:
    df = df.drop(columns=["uid"])

df = df.rename(columns={"index": "uid"})

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset shape: (361, 12)
Columns: ['uid', 'case_code', 'case_text', 'doc', 'note', 'original_text', 'reference_summary', 'title_code', 'title_text', 'urls', 'tldr_code', 'tldr_text']


,uid,case_code,case_text,doc,note,original_text,reference_summary,title_code,title_text,urls,tldr_code,tldr_text
0,tosdr001,"1,s",This service does not track you,Privacy Policy,,search encrypt does not track search history i...,this service does not track you.,"2,s",no tracking,{'searchencrypt.com'},NaN,NaN
1,tosdr002,NaN,NaN,Privacy Policy,,we also provide you additional data control op...,you can request access and deletion of persona...,"1,s",you can request access and deletion of persona...,{'stackoverflow.com'},NaN,NaN
2,tosdr003,"2,s",The copyright license that users grant to the ...,Additional Terms of Service,,rvices you grant oath the following worldwide ...,the copyright license granted to yahoo for pho...,"3,d","Yahoo's copyright license for photos, graphics...",{'flickr.com'},1,The copyright license granted to Yahoo for pho...
3,tosdr004,NaN,NaN,Terms and Conditions,,we may change these terms and conditions to re...,if you are a subscriber jagex will treat the f...,"2,s",Terms may be changed any time at their discret...,{'jagex.com'},1,"If you are a subscriber, Jagex will treat the ..."
4,tosdr005,"1,s",The service uses your personal data to employ ...,Privacy Policy,,it also enables us to serve you advertising an...,the service uses your personal data to employ ...,"2,d",targeted third-party advertising,{'academia.edu'},NaN,NaN


In [24]:
# =========================
# 4. Create extractive dataset
# =========================
# Select relevant columns and remove low-quality or missing samples

df_extractive = df[["uid", "original_text", "reference_summary"]].copy()

# Remove rows with missing values
df_extractive = df_extractive.dropna(subset=["original_text", "reference_summary"])

# Clean basic formatting
df_extractive["original_text"] = df_extractive["original_text"].astype(str).str.strip()
df_extractive["reference_summary"] = df_extractive["reference_summary"].astype(str).str.strip()

# Remove very short or low-quality examples
df_extractive = df_extractive[
    (df_extractive["original_text"].str.len() >= 50) &
    (df_extractive["reference_summary"].str.len() >= 10)
].reset_index(drop=True)

print("Final extractive dataset size:", len(df_extractive))
display(df_extractive.head())

Final extractive dataset size: 357


,uid,original_text,reference_summary
0,tosdr001,search encrypt does not track search history i...,this service does not track you.
1,tosdr002,we also provide you additional data control op...,you can request access and deletion of persona...
2,tosdr003,rvices you grant oath the following worldwide ...,the copyright license granted to yahoo for pho...
3,tosdr004,we may change these terms and conditions to re...,if you are a subscriber jagex will treat the f...
4,tosdr005,it also enables us to serve you advertising an...,the service uses your personal data to employ ...


In [25]:
# =========================
# 5. Basic text cleaning
# =========================
# Perform simple cleaning such as removing extra spaces

def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

df_extractive["clean_text"] = df_extractive["original_text"].apply(clean_text)
df_extractive["clean_summary"] = df_extractive["reference_summary"].apply(clean_text)

In [26]:
# =========================
# 6. Sentence tokenisation
# =========================
# Split documents into sentences for extractive summarisation


def get_sentences(text):
    sentences = sent_tokenize(text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
    return sentences

df_extractive["sentences"] = df_extractive["clean_text"].apply(get_sentences)

df_extractive = df_extractive[df_extractive["sentences"].apply(len) > 0].reset_index(drop=True)

print("Rows after sentence filtering:", len(df_extractive))
display(df_extractive[["clean_text", "clean_summary", "sentences"]].head())

Rows after sentence filtering: 357


,clean_text,clean_summary,sentences
0,search encrypt does not track search history i...,this service does not track you.,[search encrypt does not track search history ...
1,we also provide you additional data control op...,you can request access and deletion of persona...,[we also provide you additional data control o...
2,rvices you grant oath the following worldwide ...,the copyright license granted to yahoo for pho...,[rvices you grant oath the following worldwide...
3,we may change these terms and conditions to re...,if you are a subscriber jagex will treat the f...,[we may change these terms and conditions to r...
4,it also enables us to serve you advertising an...,the service uses your personal data to employ ...,[it also enables us to serve you advertising a...


In [27]:
# =========================
# 7. TextRank extractive summariser
# =========================
# Implement TextRank to rank sentences based on similarity graph

def textrank_summary(sentences, top_k=2):
    if len(sentences) == 0:
        return ""

    if len(sentences) <= top_k:
        return " ".join(sentences)

    vectorizer = TfidfVectorizer(stop_words="english")
    sentence_vectors = vectorizer.fit_transform(sentences)

    similarity_matrix = cosine_similarity(sentence_vectors)

    graph = nx.from_numpy_array(similarity_matrix)
    scores = nx.pagerank(graph)

    ranked_sentences = sorted(
        [(scores[i], i, sentences[i]) for i in range(len(sentences))],
        reverse=True
    )

    selected = ranked_sentences[:top_k]

    # Keep original sentence order
    selected = sorted(selected, key=lambda x: x[1])

    summary = " ".join([s[2] for s in selected])
    return summary

def dynamic_top_k(sentences):
    n = len(sentences)
    if n <= 3:
        return 1
    elif n <= 8:
        return 2
    else:
        return 3

In [36]:
# =========================
# 8. Generate summaries
# =========================
# Generate extractive summaries using TextRank

df_extractive["generated_summary"] = df_extractive["sentences"].apply(
    lambda sentences: textrank_summary(
        sentences,
        top_k=dynamic_top_k(sentences)
    )
)

display(df_extractive[["clean_summary", "generated_summary"]].head())

,clean_summary,generated_summary
0,this service does not track you.,search encrypt does not track your search hist...
1,you can request access and deletion of persona...,for more information on these choices you have...
2,the copyright license granted to yahoo for pho...,this license exists only for as long as you el...
3,if you are a subscriber jagex will treat the f...,please check the terms and conditions whenever...
4,the service uses your personal data to employ ...,it also enables us to serve you advertising an...


In [37]:
# =========================
# 9. ROUGE evaluation
# =========================
# Evaluate summarisation quality using ROUGE metrics

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

def compute_rouge(reference, generated):
    scores = scorer.score(reference, generated)
    return {
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure
    }

rouge_results = df_extractive.apply(
    lambda row: compute_rouge(row["clean_summary"], row["generated_summary"]),
    axis=1,
    result_type="expand"
)

df_results = pd.concat([df_extractive, rouge_results], axis=1)

print("Average ROUGE Scores")
print("ROUGE-1:", round(df_results["rouge1"].mean(), 4))
print("ROUGE-2:", round(df_results["rouge2"].mean(), 4))
print("ROUGE-L:", round(df_results["rougeL"].mean(), 4))

Average ROUGE Scores
ROUGE-1: 0.2317
ROUGE-2: 0.0674
ROUGE-L: 0.1841


In [38]:
# =========================
# 10. Sentence Selection Parameter Analysis (top_k)
# =========================
# Analyse the effect of different top_k values on performance

topk_results = []

for k in [1, 2, 3, 4]:
    temp_df = df_extractive.copy()
    temp_df["generated_summary"] = temp_df["sentences"].apply(
        lambda sentences: textrank_summary(sentences, top_k=k)
    )

    temp_rouge = temp_df.apply(
        lambda row: compute_rouge(row["clean_summary"], row["generated_summary"]),
        axis=1,
        result_type="expand"
    )

    topk_results.append({
        "top_k": k,
        "rouge1": temp_rouge["rouge1"].mean(),
        "rouge2": temp_rouge["rouge2"].mean(),
        "rougeL": temp_rouge["rougeL"].mean()
    })

topk_results = pd.DataFrame(topk_results)
display(topk_results)

,top_k,rouge1,rouge2,rougeL
0,1,0.234476,0.064929,0.187211
1,2,0.235276,0.076247,0.184704
2,3,0.232677,0.078207,0.182649
3,4,0.228503,0.076881,0.179232


In [45]:
# =========================
# 11. Show qualitative samples
# =========================
# Randomly select a few samples from the results to manually inspect the quality of the summaries.
# This helps to compare the original text, reference summary, and generated extractive summary.

samples = df_results.sample(5, random_state=42)

for i, row in samples.iterrows():
    print("=" * 100)
    print("UID:", row["uid"])
    print("\nORIGINAL TEXT:\n", row["clean_text"])
    print("\nREFERENCE SUMMARY:\n", row["clean_summary"])
    print("\nEXTRACTIVE SUMMARY:\n", row["generated_summary"])
    print("\nROUGE-1:", round(row["rouge1"], 4))
    print("ROUGE-2:", round(row["rouge2"], 4))
    print("ROUGE-L:", round(row["rougeL"], 4))

UID: tosdr268

ORIGINAL TEXT:
 amazon reserves the right to refuse service terminate accounts terminate your rights to use amazon services remove or edit content or cancel orders in its sole discretion.

REFERENCE SUMMARY:
 amazon can cancel your account and remove content at any time.

EXTRACTIVE SUMMARY:
 amazon reserves the right to refuse service terminate accounts terminate your rights to use amazon services remove or edit content or cancel orders in its sole discretion.

ROUGE-1: 0.3158
ROUGE-2: 0.0
ROUGE-L: 0.2105
UID: tosdr049

ORIGINAL TEXT:
 information we collect about you from third party companies including but not limited to publishing partners platforms advertising platforms and partners and data aggregators which may include attributes about you and your interests as well as other games and services you use and demographic and general location information.

REFERENCE SUMMARY:
 the service may collect extra data about you through promotions.

EXTRACTIVE SUMMARY:
 informa

In [47]:
# =========================
# 12. Best and worst case analysis
# =========================
# Identify the best and worst performing samples based on ROUGE-L score.
# This helps to understand when the model performs well and where it fails.

print("\n===== BEST SAMPLE =====")
best = df_results.sort_values("rougeL", ascending=False).iloc[0]

print(best["clean_summary"])
print(best["generated_summary"])

print("\n===== WORST SAMPLE =====")
worst = df_results.sort_values("rougeL").iloc[0]

print(worst["clean_summary"])
print(worst["generated_summary"])


===== BEST SAMPLE =====
terms of service state you affirm that you are 13 years of age or older and competent to enter into the terms conditions obligations affirmations representations and warranties set forth in these terms of service and to abide by and comply with these terms of service. according to this you must be at least 13 and able to agree to the terms to use the service.
you affirm that you are 13 years of age or older and competent to enter into the terms conditions obligations affirmations representations and warranties set forth in these terms of service and to abide by and comply with these terms of service.

===== WORST SAMPLE =====
third parties may be involved in operating the service.
to perform market research or measure website usage. and to analyze traffic counts your interests and website performance.


In [49]:
# =========================
# 12. Save results
# =========================

output_path = "/content/extractive_textrank_results.csv"
df_results.to_csv(output_path, index=False)

print("Saved results to:", output_path)

df_results.to_csv(
    "/content/drive/MyDrive/extractive_textrank_results.csv",
    index=False
)

print("Saved to Google Drive.")

Saved results to: /content/extractive_textrank_results.csv
Saved to Google Drive.


---
## **Results and Analysis of top_k**

To evaluate the impact of the sentence selection parameter (top_k), experiments were conducted using values from 1 to 4.

The results show that top_k = 2 achieves the most balanced performance across ROUGE-1, ROUGE-2, and ROUGE-L. While top_k = 3 slightly improves ROUGE-2, it does not lead to overall performance improvement, and increasing top_k further to 4 results in a decline across all metrics.

However, the differences between configurations are relatively small. This can be explained by the nature of the dataset, where many input texts are short and often consist of a single or a few sentences. As a result, selecting additional sentences does not significantly change the generated summaries.

Overall, the findings suggest that top_k has a limited impact on performance in this dataset. Therefore, a dynamic top_k strategy was adopted to better adapt to varying input lengths and maintain a balance between informativeness and conciseness.